In [ ]:
#INSTALL PACKAGES
#pip install scikit-learn torch numpy pandas ollama numpy nltk

In [2]:
#CALL MODEL

#choose model
model_name = "llama3.1:8b-instruct-q8_0"

#pull mode
!ollama pull {model_name}

!ollama list

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest 
pulling cc04e85e1f86: 100% ▕██████████████████▏ 8.5 GB                         
pulling 948af2743fc7: 100% ▕██████████████████▏ 1.5 KB                         
pulling 0ba8f0e314b4: 100% ▕██████████████████▏  12 KB                         
pulling 56bb8bd477a5: 100% ▕██████████████████▏   96 B                         
pulling 4a4a958ae550: 100% ▕██████████████████▏  485 B                         
verifying sha256 digest 
writing manifest 
success 
NAME                                                  ID              SIZE      MODIFIED               
llama3.1:8b-instruct-q8_0                             b158ded76fa0    8.5 GB    Less than a second ago    
llama3.1:8b                                           46e0c10c039e    4.9 GB    2 days ago                
llama3.1:8b-instruct-q8_0_framing_ft_frame6           39e5b7

In [3]:
#IMPORT PACKAGES AND DATASETS

import pandas as pd
import numpy as np
import json
import ollama
from pathlib import Path
import nltk
from nltk.tokenize import word_tokenize
import math

#load articles for inductive coding
df = pd.read_csv('../100_relevant_articles.csv')

In [ ]:
#GENERATE INITIAL CODES

SYSTEM_PROMPT = """You're a communication researcher who is studying the news coverage of the Mpox epidemic. 
             You will be performing inductive thematic analysis. Your job to generate initial codes for articles. 
             Your expertise is crucial in identifying issue-specific frames that can sway pubic opinions and distort public discourse. 
             Avoid generic or overly broad categories. Do not assume predefined categories, it is imperative that the codes are driven by the data.
             Think carefully to determine codes that are representative of the articles but also have maximum variability between them. 

             Format your response has to be a JSON array of objects, and include nothing else in the response. Each object should contain the following keys:
             { 
                1. "Index": "A number indicating the code order",
                2. "Name": "A short name for the code (max 3 words)",
                3. "Description": "A 3-line explanation of the code",
                4. "Example": "A representative quote (max 2 sentences)". 
             }"""

USER_PROMPT = """Attached below is an article about Mpox. Identify upto 3 unique and prevelant codes from the text. Ensure that your output format adheres to the instructions provided."""

output_list = []

#sample
#df1 = df.sample(n=5, random_state=42).reset_index(drop=True)

#or annotate on the entire set
df1 = df.copy()

for i in range(len(df1)):
    #find codes for this chunk
    messages = [
            {"role": "system", 
             "content": SYSTEM_PROMPT
            },

            {
             "role": "user", 
             "content": USER_PROMPT + df1['text'][i] 
            }

                ]

    outputs = ollama.chat(model= model_name, messages= messages)
    
    #append response
    output_list.append(outputs.message.content)
    
    #track progress
    if(i%10 == 0): print(str(i) + " iterations finished")

In [ ]:
#PARSE INITIAL CODES
df_code = pd.DataFrame(columns = ["Index", "Name", "Description", "Example"])

for i in range(len(output_list)):
    codes = output_list[i]

    try:
        df_temp = pd.DataFrame(json.loads(codes), columns = ["Index", "Name", "Description", "Example"])
        df_code = pd.concat([df_code, df_temp], axis = 0) 
    
    except Exception as e: 
        continue

df_code = df_code.drop_duplicates(subset="Name", keep="first").reset_index(drop=True) #only keep unique codes
df_code['Index'] = range(len(df_code))
df_code.to_csv('llama_initial_codes.csv') #save the initial codes

In [ ]:
#CLUSTER CODES FURTHER

SYSTEM_PROMPT = """You're a communication researcher who is conducting an issue-specific narrative framing analysis on the news coverage of the Mpox epidemic. 
    A researcher has already gone over news articles and come up with a list of initial codes through inductive coding. Your job is go over a set of initial codes and cluster them to flesh out unique frames.
    In his seminal work, Entman said "Framing essentially involves selection and salience. To frame is to select some aspects of aperceived reality and make them more salient in a communicating text, in such a way as to promote a 
    particular problem definition, causal interpretation, moral evaluation, and/or treatment recommendation for the item described." Follow Entman's principles and define frames based on the 4 unique framing elements. 
    Prioritize frame clarity and uniqueness. 

    Format your response as an array of upto 3 JSON objects, and include nothing else in the response. Each object should represent a frame as follows:
    { 
        "Name": "A short name for the frame (max 3 words)",
        "Problem definition": "How do this frame define the problem at hand?",
        "Causal attribution": "What actors or forces does this frame attribute the cause of the problem to?",
        "Moral evaluation": "What value judgements are being made by this frame?"
        "Treatment recommendation": "What remedies does the frame suggest for tackling the problem?"
        "Example": "A representative quotes (max 2 lines each)". 
    }"""

USER_PROMPT = """Ensure that you adhere to the output format provided.
    Go over the attached codes and group them into frames for a framing analysis. 
    The following text contains all the identified codes as a JSON array of objects. 
    """ 

df_code = pd.read_csv('llama_initial_codes.csv')

num_chunks = 5 #process the codes in chunks since there are over 200 of them
cap = math.ceil(len(df_code)/num_chunks) #number of codes per chunk

df_frames = pd.DataFrame(columns = ["Name", "Problem definition", "Causal attribution", "Moral evaluation", "Treatment recommendation", "Example"])  #create dataframe

flist = []

for n in range(num_chunks):
    #turn the codes into a chunk
    df_temp = df_code.loc[range(n*cap, min(len(df_code), (n+1)*cap + 1)),:]
    json_obj = df_temp.to_dict(orient="records")
    codes = json.dumps(json_obj, indent=2, ensure_ascii=False)

    messages = [
                {"role": "system", 
                "content": SYSTEM_PROMPT
                },

                {
                "role": "user", 
                "content": USER_PROMPT + codes
                }

                    ]

    outputs = ollama.chat(model= model_name, messages= messages)
    flist.append(outputs.message.content)

    try:
        frames = json.loads(outputs.message.content)
        df_frames = pd.concat([df_frames, pd.DataFrame(frames)], ignore_index=True)

    except Exception as e:
        continue

    print(str(n) + " chunks processed")

#FORMAT AND SAVE OBTAINED FRAMES
df_frames.to_csv('llama_themes.csv')

1 chunks processed
3 chunks processed
4 chunks processed


In [ ]:
#REVISE FRAMES
SYSTEM_PROMPT = """You're a communication researcher who is conducting an issue-specific narrative framing analysis on the news coverage of the Mpox epidemic. 
    In his seminal work, Entman said "Framing essentially involves selection and salience. To frame is to select some aspects of aperceived reality and make them more salient in a communicating text, in such a way as to promote a 
    particular problem definition, causal interpretation, moral evaluation, and/or treatment recommendation for the item described." 
    A researcher has already gone over news articles and come up with a list of frames with Entman's framing elements. 
    Your job is to go over all the frames, group them based on conceptual similarity and produce a revised set of frames with upto 10 unique frames. 
    Follow Entman's principles and define frames based on the 4 unique framing elements. 
    Take your time to think carefully and prioritize frame clarity and uniqueness. 

    Format your response as an array of upto 10 JSON objects, and include nothing else in the response. Each object should represent a frame as follows:
    { 
        "Name": "A short name for the frame (max 3 words)",
        "Description": "2-3 lines of detailed description of the frame",
        "Problem definition": "How do this frame define the problem at hand?",
        "Causal attribution": "What actors or forces does this frame attribute the cause of the problem to?",
        "Moral evaluation": "What value judgements are being made by this frame?"
        "Treatment recommendation": "What remedies does the frame suggest for tackling the problem?"
        "Example": "A representative quotes (max 2 lines each)". 
    }"""

USER_PROMPT = """Go over the attached set of frames and produce a revised set of upto 10 unique frames for a codebook based annotation procedure. 
    The following text contains all the identified frames as a JSON array of objects. 
    """ 

#load data
df_frames = pd.read_csv('llama_themes.csv')
df_frames = df_frames.iloc[:,1:]
json_obj = df_frames.to_dict(orient="records")
frames = json.dumps(json_obj, indent=2, ensure_ascii=False)

messages = [
                {"role": "system", 
                "content": SYSTEM_PROMPT
                },

                {
                "role": "user", 
                "content": USER_PROMPT + frames
                }

            ]

outputs = ollama.chat(model= model_name, messages= messages)

frames_revised = json.loads(outputs.message.content)
df_frames_revised = pd.DataFrame(frames_revised)
df_frames_revised.to_csv('llama_frames.csv', index=False)